# Sample-Level OOV Analysis of Chemical Formulae

This notebook analyzes the **sample-level out-of-vocabulary (OOV) rate of chemical formulae (CFs)** across different dataset splits. The goal is to quantify how often peaks in test spectra contain chemical formulae that **never appeared in the training set**, and to compare this behavior across different split strategies.

Understanding OOV rates is important because many machine learning models implicitly assume that the test distribution is similar to the training distribution. If a large fraction of peaks in the test set correspond to chemical formulae that were never observed during training, model performance may degrade due to distribution shift.

In particular, we hypothesize that the LS split is a more challenging split precisely because of this reason

# Overview of the Pipeline

The notebook proceeds in three steps:

### Step 1: Cache training-set chemical formulae
For each dataset and split, we extract and cache the **set of chemical formulae that appear in the training spectra**.  
This step avoids repeatedly recomputing the same information and speeds up later analysis.

### Step 2: Compute sample-level OOV rate
For each test spectrum, we compute the **sample-level OOV rate**, defined as the proportion of peaks whose chemical formulae do **not appear in the training set**.

Formally, for a test spectrum \(i\):

$$
\text{OOV}_i =
\frac{\# \text{peaks with CF not in training set}}{\# \text{peaks in spectrum}}
$$

These values are cached for later statistical analysis.



### Step 3: Compare OOV rates across splits
Finally, we compare the sample-level OOV distributions between different dataset splits.  
In particular, we test whether the **LS split produces higher OOV rates** than other splits (e.g., random or scaffold splits).

We perform a **Welch's t-test** to compare the distributions:

$$
H_0: \mu_{\text{LS}} \le \mu_{\text{split}}
$$

$$
H_1: \mu_{\text{LS}} > \mu_{\text{split}}
$$

This evaluates whether the LS split leads to a statistically higher fraction of unseen chemical formulae.

# Outputs

For each dataset and split, the notebook reports:

- mean sample-level OOV rate
- statistical significance of the test


Import libraries

In [ ]:
import os 
import numpy as np
from tqdm import tqdm
from scipy import stats
from pathlib import Path
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind
from utils import load_pickle, pickle_data, load_json

import torch

Settings

In [11]:
data_folder = Path("../data/processed_data")
splits_folder = Path("../data/splits")
cache_folder = Path("../results/OOV_CF_analysis")

if not os.path.exists(cache_folder): os.makedirs(cache_folder)

datasets = ["NPLIB1", "massspecgym"]
splits = ["random", "scaffold", "LS"]

Helper Functions

In [37]:
def get_CFs_train(data, train_ids):

    train_CFs = set() 

    for rec in tqdm(data): 

        if str(rec["id_"]) not in train_ids: continue 

        CFs = set([f["comment"]["f_pred"] for f in rec["peaks"]])
        CFs.discard("")
        train_CFs.update(CFs)

    return train_CFs

def compute_sample_level_OOV_rate(train_CFs, data, test_ids):

    oov_rate_list = []

    for rec in tqdm(data):

        if str(rec["id_"]) not in test_ids: continue 

        CFs = set([f["comment"]["f_pred"] for f in rec["peaks"]])
        CFs.discard("")

        if len(CFs) == 0:
            oov_rate = 1.0
        else:
            oov_rate = np.mean([f not in train_CFs for f in CFs]).item()
        
        oov_rate_list.append(oov_rate)

    return oov_rate_list

Step 1: Cache the chemical formula (CFs) of all the peaks in the training set

In [38]:
for dataset in datasets:

    for split in splits: 
        
        output_folder = cache_folder / dataset / split
        output_path = output_folder / "train_CFs.pkl"
        if os.path.exists(output_path): 
            print(f"Cache for CF of training samples for {dataset} ({split} split) created. Skipping.")
            continue

        # Create the folder 
        if not os.path.exists(output_folder): os.makedirs(output_folder)

        # Load the data 
        data_path = data_folder / f"{dataset}.pkl"
        data = load_pickle(data_path)
        print(f"Loaded data: {dataset}")

        # Load the split 
        split_ids = load_json(splits_folder / dataset / f"{split}.json")
        train_ids = [t.replace(".pkl", "") for t in split_ids["train"]]

        # Get CFs that appeared in the training set
        train_CFs = get_CFs_train(data, train_ids)

        # Save the data 
        pickle_data(train_CFs, output_path)
        print(f"Got all the chemical formula that appeared in the training set for {dataset} ({split} split).")

Cache for CF of training samples for NPLIB1 (random split) created. Skipping.
Cache for CF of training samples for NPLIB1 (scaffold split) created. Skipping.
Cache for CF of training samples for NPLIB1 (LS split) created. Skipping.
Cache for CF of training samples for massspecgym (random split) created. Skipping.
Cache for CF of training samples for massspecgym (scaffold split) created. Skipping.
Cache for CF of training samples for massspecgym (LS split) created. Skipping.


Step 2: Compute and cache sample level OOV rate

In [ ]:
for dataset in datasets:

    for split in splits: 
        
        output_folder = cache_folder / dataset / split
        output_path = output_folder / "test_sample_level_OOV_rate.pkl"
        if os.path.exists(output_path): 
            print(f"Cache for sample level oov rate for the test samples for {dataset} ({split} split) created. Skipping.")
            continue

        # Get the training level OOV rate
        train_CFs_path = output_folder / "train_CFs.pkl"
        if not os.path.exists(train_CFs_path):
            raise Exception(f"Make sure to cache the train CF by running the previous cell.")
        train_CFs = load_pickle(train_CFs_path)

        # Load the data 
        data_path = data_folder / f"{dataset}.pkl"
        data = load_pickle(data_path)
        print(f"Loaded data: {dataset}")

        # Load the split 
        split_ids = load_json(splits_folder / dataset / f"{split}.json")
        test_ids = [t.replace(".pkl", "") for t in split_ids["test"]]

        # Get sample level OOV rate 
        sample_level_oov = compute_sample_level_OOV_rate(train_CFs, data, test_ids)
        pickle_data(sample_level_oov, output_path)


Cache for sample level oov rate for the test samples for NPLIB1 (random split) created. Skipping.
Cache for sample level oov rate for the test samples for NPLIB1 (scaffold split) created. Skipping.
Cache for sample level oov rate for the test samples for NPLIB1 (LS split) created. Skipping.
Cache for sample level oov rate for the test samples for massspecgym (random split) created. Skipping.
Cache for sample level oov rate for the test samples for massspecgym (scaffold split) created. Skipping.
Cache for sample level oov rate for the test samples for massspecgym (LS split) created. Skipping.


Step 3: Compare the sample level OOV rate against the LS split

In [45]:
for dataset in datasets:

    for split in splits: 
        
        if split == "LS": continue 

        # Get the sample level OOV rate for the LS split 
        LS_sample_level_oov_rate_path = output_path = cache_folder / dataset / "LS" / "test_sample_level_OOV_rate.pkl"
        if not os.path.exists(LS_sample_level_oov_rate_path):
            raise Exception(f"Make sure to get the sample level OOV rate for the LS split by running the previous cell.")
        LS_sample_level_oov_rate = load_pickle(LS_sample_level_oov_rate_path)
        
        # Get the sample level OOV rate for the current split 
        sample_level_oov_rate_path = output_path = cache_folder / dataset / split / "test_sample_level_OOV_rate.pkl"
        if not os.path.exists(sample_level_oov_rate_path):
            raise Exception(f"Make sure to get the sample level OOV rate for the {split} split by running the previous cell.")
        sample_level_oov_rate = load_pickle(sample_level_oov_rate_path)
    
        # Compute Welch's t-test
        t, p = ttest_ind(LS_sample_level_oov_rate, sample_level_oov_rate, equal_var=False, alternative="greater") # Check if this is higher in the LS split

        # Look at the oov rate
        print(dataset, split, round(np.mean(sample_level_oov_rate) * 100, 3), p)


NPLIB1 random 4.004 8.623308505512745e-24
NPLIB1 scaffold 5.097 4.7421329419059477e-11
massspecgym random 2.2 0.0
massspecgym scaffold 7.445 1.7085959680296347e-42
